# 1. PAIRED β REPLICATION — AeroSync-Mamba v17

v17 starts from the v16 reference notebook. It asks one question: **does LACA's lag bias (β = 1.0) really beat
no lag bias (β = 0.0)?** The v9 ablation found **+0.0057 macro-F1** on a single seed (42). That result was
accepted against the ±0.0005 same-seed placeholder. The measured seed-to-seed std is **0.0160**, almost 3× larger
than the effect.

**Design: paired, 10 runs.** β ∈ {0.0, 1.0} × seed ∈ {42, 123, 2024, 7, 99}. These are the same five seeds as
v15/v16, so the β = 1.0 arm can be compared with existing data. Runs go in pairs: for each seed, β = 0.0 and then
β = 1.0. If the session stops partway, every finished seed still has both conditions.

Each run resets every RNG with its seed and builds a fresh model. β is a plain float multiplier. It creates no
parameters and uses no random numbers. So for a given seed, both β values start from **identical initial weights**
and get **identical shuffle and dropout streams**. The notebook checks this for every pair. The only difference is
whether the lag bias is added to the attention scores. Seed-specific effects, such as seed 99's weaker solution,
therefore cancel within each pair.

**Everything else is fixed at the v16 values:** strongest-64 audio, sqrt, no normalization, no window, Ta = 64;
raw 6-channel sensor; mlp g(Δτ) (97 params); 4 heads; Mamba 2 × (d_state 16, d_conv 2); λ_align = λ_cons = 0.1;
class-weighted CE.

**Analysis (Section 7).** A per-seed paired table for all four metrics, with macro-F1 as the primary metric. It
reports the mean paired difference, the std **of the paired differences**, the win count, a paired t-test (primary)
and a Wilcoxon signed-rank test (secondary), then a plain verdict.

**Reference checks.** Seed 42 at β = 1.0 must reproduce the v16 baseline: accuracy 0.9080, macro-F1 0.9048,
balanced accuracy 0.9022, macro-AUROC 0.9894 (±0.0005). The notebook checks this right after run 2 and stops if it
fails. Seed 42 at β = 0.0 is compared with v9's 0.8909. That comparison is informational only: v9's own β = 1.0
baseline was 0.8966, not 0.9048, because the configuration changed after v9 (v12 Mamba configuration). So a gap
there is expected and does not indicate an error.

| Decision | Value | Evidence status |
|---|---|---|
| Timestamps into HTT / LACA | Real per-window audio and IMU timestamps (not a synthetic `linspace`) | CORRECTNESS FIX (earlier versions passed an identical synthetic grid for every window) |
| Fusion backbone | Real `mamba_ssm.Mamba` selective-scan SSM; pure-PyTorch fallback only when no CUDA GPU is available (printed unmissably) | CORRECTNESS FIX (the earlier gated-conv block was not an SSM) |
| Audio bin selection | Strongest-N FFT bins by mean magnitude, computed once from **training** audio only, then frozen and reused for val/test | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): strongest-64 0.8736 vs first-64 0.8494, +0.0242 (bin-selection study) |
| Frequency budget N | 64 bins | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v6: N=64 0.8966 vs N=48 0.8863, **+0.0103**, which is below 0.0160. The other budgets clear it (N=32 +0.0249, N=128 +0.0166) except N=96 (+0.0142). |
| Audio compression | `sqrt(|rFFT|)` | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 sqrt 0.8966 vs log_db 0.8696 (+0.0270) and log1p 0.8649 (+0.0317) |
| Audio windowing | None; Hann windowing rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 no window 0.8649 vs Hann 0.8290 (+0.0359) |
| Audio normalization | None; z-score normalization rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v4 none 0.8966 vs z-score 0.8684 (+0.0282) |
| Audio time positions Ta | 64 per 2.56 s window | PROVISIONAL: v7 Ta=128 0.8967 vs Ta=64 0.8966 (−0.0001, a tie). The cheaper Ta=64 is kept. |
| Sensor input | Raw 6 channels (accel x/y/z, gyro x/y/z), Ts=128; derived magnitude features rejected | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v8 raw6 0.8966 vs best derived set 0.8740 (+0.0226) |
| LACA temporal-bias scale β | 1.0 | **UNDER TEST IN THIS NOTEBOOK** (paired 5-seed replication of β=0.0 vs β=1.0). Previously PROVISIONAL: v9 single seed, β=1 0.8966 vs β=0 0.8909, +0.0057, below the 0.0160 seed-to-seed std. |
| LACA lag-bias form g(Δτ) | mlp `Linear(1,32) → Tanh → Linear(32,1)`, 97 params | PROVISIONAL: v10 mlp vs rbf +0.0029, vs bucketed +0.0030 (below 0.0160) |
| LACA heads | 4 | PROVISIONAL: v11 h=1 was +0.0008 (one test window) and h=4 was kept. h=2/8/16 were 0.006–0.012 worse (below 0.0160). |
| Mamba fusion | 2 blocks, d_state=16, d_conv=2 (expand 2) | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v12: 2 vs 4 blocks +0.0023, d_state 16 vs 8 +0.0172, d_conv 2 vs 4 +0.0059. |
| λ_align (InfoNCE) | 0.1 | PROVISIONAL: v13 removing L_align cost −0.0088, which is below 0.0160. λ=0.05 was −0.0271 and λ=0.2 was −0.0070. |
| λ_cons (KL consistency) | 0.1 | **PROVISIONAL** (margin exceeded the old same-seed placeholder ±0.0005 but has not yet been verified against the true measured seed-to-seed std of 0.0160). v13 grid: λ=0.2 +0.0008, λ=0.05 −0.0232. Removing L_cons cost −0.0165. |
| Ordinal-severity loss | Not used (rejected) | **REAL** (margin exceeds measured seed-to-seed std of 0.0160): v14 best λ_ord>0 0.8859 vs none 0.9048 (−0.0189); adjacent-class errors rose 94 → 113 |
| Classification loss | Cross-entropy weighted by inverse **training-set** class frequency | CORRECTNESS FIX (class-imbalance handling) |
| Seeding | Python `random`, numpy, torch CPU + all CUDA devices, cuDNN deterministic; reset per seed at run start and right before model construction. The mission split has its own fixed seed (42), so every seed sees the same train/val/test windows. | CORRECTNESS FIX (reproducibility) |

## 2. Configuration constants & environment
Every fixed value lives in the first cell below. The second installs `mamba_ssm` and reports which fusion backbone is active.

In [1]:
import copy
import csv
import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import subprocess
import sys
import time
import urllib.request
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

# ==============================================================================
# FIXED CONFIGURATION — every finalized choice (see the summary table above)
# ==============================================================================
# --- Paired beta replication: beta x seed ---
SEEDS = [42, 123, 2024, 7, 99]           # same 5 seeds as v15/v16, in this order
BETA_VALUES = [0.0, 1.0]                 # per seed, in this order (0.0 = lag-bias ablation, 1.0 = finalized value)
RUN_ORDER = [(seed, beta) for seed in SEEDS for beta in BETA_VALUES]   # 5 back-to-back pairs = 10 runs
SPLIT_SEED = 42                          # mission-level train/val/test split (independent of the training seed)

# --- Dataset / split ---
DATASET_REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"
WORK_DIR = Path("/kaggle/working")
DATASET_BASE = WORK_DIR / "tii_uav_dataset"
DATASET_ROOT = DATASET_BASE / "Dataset"
TRAIN_FRACTION = 0.70
VAL_FRACTION = 0.15
N_CLASSES = 5                            # propeller-fault severity 0..4

# --- Windowing ---
WINDOW_SEC = 2.56
STEP_SEC = 1.28
MIN_SENSOR_SAMPLES_PER_WINDOW = 10

# --- Audio: strongest-64 bins (train-only), sqrt(|rFFT|), no window, no normalization ---
N_AUDIO_BINS = 64                        # frequency axis
TA_AUDIO = 64                            # audio time positions per window

# --- Sensor: raw 6 channels (accel x/y/z, gyro x/y/z), no derived features ---
N_SENSOR_CHANNELS = 6
TS_SENSOR = 128                          # sensor time positions per window
SENSOR_STD_EPS = 1e-6

# --- Model ---
D_MODEL = 256
TIME_EMBED_FREQUENCIES = 64
LACA_NUM_HEADS = 4
LACA_BETA = 1.0                          # finalized default; each run passes its own beta explicitly
LAG_BIAS_HIDDEN = 32                     # mlp g(Δτ): Linear(1,32) -> Tanh -> Linear(32,1) = 97 params
LAG_BIAS_PARAMS = 97
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496          # 2 x 437,248 (real mamba_ssm)
DROPOUT = 0.15

# --- Loss: L_cls (class-weighted CE) + 0.1 * L_align (InfoNCE) + 0.1 * L_cons (KL) ---
LAMBDA_ALIGN = 0.1
LAMBDA_CONS = 0.1
INFONCE_TEMPERATURE = 0.07

# --- Optimization ---
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2

# --- Outputs ---
CHECKPOINT_DIR = WORK_DIR / "v17_checkpoints"
PROBS_DIR = WORK_DIR / "v17_test_probs"
RESULTS_CSV = WORK_DIR / "beta_paired_replication_results.csv"
for _d in (CHECKPOINT_DIR, PROBS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- Reference numbers ---
REFERENCE_SEED42 = {"accuracy": 0.9080, "macro_f1": 0.9048, "balanced_accuracy": 0.9022, "macro_auroc": 0.9894}  # beta=1.0
SAME_SEED_TOLERANCE = 0.0005             # same-seed reproducibility (code correctness), not a finding threshold
HALT_ON_BASELINE_MISMATCH = True         # stop after run 2 if seed 42 / beta=1.0 misses REFERENCE_SEED42
V9_SEED42_MACRO_F1 = {0.0: 0.8909, 1.0: 0.8966}   # v9 single-seed ablation (v7-era config; informational only)
V9_SINGLE_SEED_BETA_EFFECT = 0.0057      # v9: beta=1 minus beta=0, seed 42 only
SEED_TO_SEED_STD_F1 = 0.0160             # v15/v16 measured seed-to-seed std of test macro-F1 (beta=1.0)
V16_BETA1_MACRO_F1 = {42: 0.9047540152267212, 123: 0.8818976370764308, 2024: 0.8783309161428443,
                      7: 0.9020733412115728, 99: 0.8674825744048924}   # v16_seed_results.csv (informational)
PAIRED_ALPHA = 0.05
REFERENCE_SEED42_RNG_PREFIX = "5641b401720d"   # torch CPU RNG right after seed-42 model construction (v10-v15)
REFERENCE_DATA = {
    "bin_indices": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255,
                    258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508,
                    509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760,
                    761, 762, 763, 764, 765, 766],
    "sensor_fingerprint": {
        "train": (4934, -6167105.225, 808427.456, (906, 1020, 1004, 1008, 996)),
        "val":   (954, -1193206.652, 156301.664, (126, 229, 199, 190, 210)),
        "test":  (1130, -1413563.944, 185127.438, (286, 205, 210, 228, 201)),
    },
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")


def set_all_seeds(seed: int) -> None:
    """Seed python `random`, numpy, torch CPU and every CUDA device; deterministic cuDNN."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


print(f"Paired beta replication: beta {BETA_VALUES} x seeds {SEEDS} = {len(RUN_ORDER)} runs | results -> {RESULTS_CSV}")
print(f"Fixed configuration: audio strongest-{N_AUDIO_BINS}, Ta={TA_AUDIO}, sqrt, no window, no normalization "
      f"| sensor raw {N_SENSOR_CHANNELS} ch, Ts={TS_SENSOR} | LACA beta=<per run>, {LACA_NUM_HEADS} heads, mlp g(Δτ) "
      f"{LAG_BIAS_PARAMS} params | Mamba {MAMBA_NUM_BLOCKS} x (d_state {MAMBA_D_STATE}, d_conv {MAMBA_D_CONV}) | "
      f"loss CE + {LAMBDA_ALIGN} InfoNCE + {LAMBDA_CONS} KL | {EPOCHS} epochs, batch {BATCH_SIZE}, AdamW {LEARNING_RATE}")

Device: cuda (Tesla T4)
Paired beta replication: beta [0.0, 1.0] x seeds [42, 123, 2024, 7, 99] = 10 runs | results -> /kaggle/working/beta_paired_replication_results.csv
Fixed configuration: audio strongest-64, Ta=64, sqrt, no window, no normalization | sensor raw 6 ch, Ts=128 | LACA beta=<per run>, 4 heads, mlp g(Δτ) 97 params | Mamba 2 x (d_state 16, d_conv 2) | loss CE + 0.1 InfoNCE + 0.1 KL | 20 epochs, batch 32, AdamW 0.0001


In [2]:
# ==============================================================================
# mamba_ssm installation + fusion-backbone resolution.
# Order: an already-installed package, else a prebuilt wheel matched to this
# Python / torch / CUDA / C++ ABI (the reliable path on Kaggle), else a
# time-capped source build. If none succeeds, or there is no CUDA GPU, the
# pure-PyTorch fallback block is used and announced below.
# ==============================================================================
PIP_TIMEOUT_S = 240


def _env_tags():
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    try:
        cxx11_abi = bool(torch._C._GLIBCXX_USE_CXX11_ABI)
    except AttributeError:
        cxx11_abi = False
    return py_tag, torch_mm, torch.version.cuda, cxx11_abi


def _prebuilt_wheel_url(repo, py_tag, torch_mm, cuda_ver, cxx11_abi):
    """Best-matching .whl asset in `repo`'s GitHub releases (exact CUDA version first, then CUDA major)."""
    req = urllib.request.Request(f"https://api.github.com/repos/{repo}/releases",
                                 headers={"User-Agent": "kaggle-notebook", "Accept": "application/vnd.github+json"})
    try:
        with urllib.request.urlopen(req, timeout=20) as resp:
            releases = json.load(resp)
    except Exception as exc:
        print(f"    could not query github.com/{repo}/releases: {exc!r}")
        return None
    abi = f"cxx11abi{'TRUE' if cxx11_abi else 'FALSE'}"
    cuda_exact = f"cu{cuda_ver.replace('.', '')}"
    cuda_major = f"cu{cuda_ver.split('.')[0]}"
    exact, major_only = [], []
    for rel in releases:
        for asset in rel.get("assets", []):
            name = asset.get("name", "")
            if not (name.endswith(".whl") and py_tag in name and abi in name and f"torch{torch_mm}" in name):
                continue
            if cuda_exact in name:
                exact.append(asset["browser_download_url"])
            elif cuda_major in name:
                major_only.append(asset["browser_download_url"])
    chosen = exact or major_only
    return chosen[0] if chosen else None


def _pip_install(args):
    """pip with a hard timeout and captured output, so a stuck build can never hang the kernel."""
    try:
        proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                              timeout=PIP_TIMEOUT_S, capture_output=True, text=True, check=False)
    except subprocess.TimeoutExpired:
        print(f"    pip install {args} exceeded {PIP_TIMEOUT_S}s — killed")
        return False
    if proc.returncode != 0:
        print(f"    pip install {args} failed (exit {proc.returncode}): {(proc.stderr or proc.stdout)[-400:]}")
    return proc.returncode == 0


def install_mamba_ssm():
    if importlib.util.find_spec("mamba_ssm") is not None:
        return "already installed"
    tags = _env_tags()
    print(f"Installing mamba_ssm + causal-conv1d (python={tags[0]}, torch={tags[1]}, cuda={tags[2]}, cxx11abi={tags[3]})")
    urls = [_prebuilt_wheel_url(repo, *tags) for repo in ("Dao-AILab/causal-conv1d", "state-spaces/mamba")]
    if all(urls) and all(_pip_install([u, "--no-deps"]) for u in urls):
        return "prebuilt wheel"
    print("  no complete prebuilt-wheel match — trying a time-capped source build")
    if _pip_install(["causal-conv1d>=1.2.0", "--no-build-isolation"]) and _pip_install(["mamba-ssm", "--no-build-isolation"]):
        return "source build"
    return "failed"


MAMBA_CLASS = None
if torch.cuda.is_available():
    print(f"mamba_ssm install: {install_mamba_ssm()}")
    try:
        from mamba_ssm import Mamba as MAMBA_CLASS
    except ImportError as exc:
        print(f"mamba_ssm import failed: {exc!r}")

if MAMBA_CLASS is not None:
    FUSION_BACKBONE = "mamba_ssm.Mamba (real selective-scan SSM, CUDA)"
    print(f"✅ FUSION BACKBONE: {FUSION_BACKBONE}")
else:
    FUSION_BACKBONE = "FALLBACK pure-PyTorch gated-conv block (NOT a state-space model)"
    print("🚨" * 45)
    print(f"🚨  FUSION BACKBONE: {FUSION_BACKBONE}")
    print("🚨  " + ("no CUDA GPU detected" if not torch.cuda.is_available() else "mamba_ssm could not be installed/imported")
          + " — results will NOT match the reference numbers, which were produced with real mamba_ssm.")
    print("🚨" * 45)

Installing mamba_ssm + causal-conv1d (python=cp312, torch=2.10, cuda=12.8, cxx11abi=True)
    pip install ['https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.7.0/causal_conv1d-1.7.0%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl', '--no-deps'] failed (exit 1): ERROR: causal_conv1d-1.7.0+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_aarch64.whl is not a supported wheel on this platform.

  no complete prebuilt-wheel match — trying a time-capped source build
mamba_ssm install: source build
✅ FUSION BACKBONE: mamba_ssm.Mamba (real selective-scan SSM, CUDA)


## 3. Data loading & preprocessing
Mission-level split, strongest-64 bin selection from training audio only, 2.56 s windows with real timestamps, and sensor standardization with training-split statistics. Leakage and timestamp checks are included.

In [3]:
# ==============================================================================
# DATA — TII UAV Realistic Fault Dataset: parse missions -> mission-level split
# -> strongest-64 bins from TRAINING audio only (frozen) -> 2.56 s windows with
# real per-window timestamps -> datasets (sensor standardized with TRAINING stats).
# ==============================================================================
if not DATASET_ROOT.exists():
    print(f"Cloning {DATASET_REPO_URL} ...")
    subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO_URL, str(DATASET_BASE)], check=True)


def parse_mission(mission_dir: Path):
    """Real 6-axis IMU (seconds from mission start) + raw audio chunks with per-chunk timestamps.
    Audio chunk duration = the mission's IMU-measured duration / number of chunks (both come from the same
    continuous recording; the audio log carries no per-chunk timestamp)."""
    sensor_files = list(mission_dir.glob("*SensorCombined*.jsonl"))
    audio_files = list(mission_dir.glob("*AudioBuffer*.jsonl"))
    if not sensor_files or not audio_files:
        return None

    sensor_times, accel, gyro = [], [], []
    with open(sensor_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                t_sec = float(rec["timestamp"]) / 1e6
                sensor_times.append(t_sec)
                accel.append(rec.get("accelerometer_m_s2", [0.0, 0.0, 9.81]))
                gyro.append(rec.get("gyro_rad", [0.0, 0.0, 0.0]))
            except Exception:
                continue
    if len(sensor_times) < 50:
        return None
    sensor_times = np.array(sensor_times, dtype=np.float64)
    sensor_times = sensor_times - sensor_times[0]
    telemetry = np.concatenate([np.array(accel, dtype=np.float32), np.array(gyro, dtype=np.float32)], axis=-1)

    audio_chunks = []
    with open(audio_files[0], "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
                chunk = rec.get("audio_data", rec.get("buffer", rec.get("data", [])))
                if len(chunk) > 0:
                    audio_chunks.append(chunk)
            except Exception:
                continue
    if len(audio_chunks) < 10:
        return None
    audio_chunks = np.array(audio_chunks, dtype=np.float32)

    chunk_duration_s = (sensor_times[-1] - sensor_times[0]) / len(audio_chunks)
    audio_times = np.arange(len(audio_chunks), dtype=np.float64) * chunk_duration_s
    return {"sensor_times": sensor_times, "telemetry": telemetry, "audio_times": audio_times, "audio_chunks": audio_chunks}


def split_missions(dataset_root: Path):
    """Mission-level stratified split (70 / 15 / 15 per class) with its own fixed seed."""
    class_missions = defaultdict(list)
    for class_dir in sorted(dataset_root.iterdir()):
        if not class_dir.is_dir() or class_dir.name.startswith("."):
            continue
        try:
            label = int(class_dir.name)
        except ValueError:
            continue
        for mission_dir in sorted(class_dir.iterdir()):
            if mission_dir.is_dir() and not mission_dir.name.startswith("."):
                class_missions[label].append(mission_dir)

    random.seed(SPLIT_SEED)
    splits = {"train": {}, "val": {}, "test": {}}
    for label in sorted(class_missions):
        missions = class_missions[label]
        random.shuffle(missions)
        n_train = max(1, int(len(missions) * TRAIN_FRACTION))
        n_val = max(1, int(len(missions) * VAL_FRACTION))
        splits["train"][label] = missions[:n_train]
        splits["val"][label] = missions[n_train:n_train + n_val]
        splits["test"][label] = missions[n_train + n_val:]
        print(f"Class {label} ({len(missions)} flights) -> train {n_train}, val {len(splits['val'][label])}, "
              f"test {len(splits['test'][label])}")
    return splits


def select_strongest_bins(train_missions, n_bins):
    """Indices of the n_bins rFFT magnitude bins with the highest mean magnitude over TRAINING audio only
    (unsupervised energy, no labels). Returned sorted; frozen and reused for val/test."""
    magnitude_sum, n_chunks, fft_len = None, 0, None
    for mission in train_missions:
        mag = np.abs(np.fft.rfft(mission["audio_chunks"], axis=1))
        fft_len = mag.shape[1] if fft_len is None else min(fft_len, mag.shape[1])
        mag = mag[:, :fft_len]
        if magnitude_sum is not None and len(magnitude_sum) > fft_len:
            magnitude_sum = magnitude_sum[:fft_len]
        magnitude_sum = mag.sum(axis=0) if magnitude_sum is None else magnitude_sum + mag.sum(axis=0)
        n_chunks += mag.shape[0]
    mean_magnitude = magnitude_sum / n_chunks
    return np.sort(np.argsort(mean_magnitude)[::-1][:n_bins]).astype(int)


def audio_features(audio_chunks, bin_indices):
    """raw chunk -> rFFT (no window) -> |.| -> frozen strongest bins -> sqrt (no normalization)."""
    mag = np.abs(np.fft.rfft(audio_chunks, axis=1))
    valid = bin_indices[bin_indices < mag.shape[1]]
    feats = np.zeros((mag.shape[0], len(bin_indices)), dtype=np.float32)
    feats[:, :len(valid)] = np.sqrt(mag[:, valid]).astype(np.float32)
    return feats


def slice_windows(mission, label, bin_indices):
    """2.56 s windows (1.28 s step). Sensor: TS_SENSOR real IMU samples chosen evenly by index inside the window;
    audio: TA_AUDIO real chunk timestamps chosen the same way (a window with fewer than 2 audio chunks — the
    recording edge — falls back to an even grid). tau_a / tau_s = those REAL timestamps relative to window start."""
    s_times, telemetry = mission["sensor_times"], mission["telemetry"]
    a_times = mission["audio_times"]
    a_feats = audio_features(mission["audio_chunks"], bin_indices)
    max_time = min(s_times[-1], a_times[-1])

    windows, t_start = [], 0.0
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue

        s_idx = np.where(s_mask)[0]
        tau_s_abs = s_times[s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]]
        x_s = np.zeros((TS_SENSOR, N_SENSOR_CHANNELS), dtype=np.float32)
        for c in range(N_SENSOR_CHANNELS):
            x_s[:, c] = np.interp(tau_s_abs, s_times[s_mask], telemetry[s_mask, c])

        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            tau_a_abs = a_times[a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]]
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
        x_a = np.zeros((TA_AUDIO, len(bin_indices)), dtype=np.float32)
        for c in range(len(bin_indices)):
            x_a[:, c] = np.interp(tau_a_abs, a_times, a_feats[:, c], left=0.0, right=0.0)

        windows.append({
            "x_a": x_a,
            "x_s": x_s,
            "label": int(label),
            "tau_a": (tau_a_abs - t_start).astype(np.float32),
            "tau_s": (tau_s_abs - t_start).astype(np.float32),
            "window_sec": np.float32(WINDOW_SEC),
        })
        t_start += STEP_SEC
    return windows


class UAVWindowDataset(Dataset):
    """Windows -> tensors. Sensor channels are standardized with mean/std that must come from the TRAINING
    split (computed here only when none are passed, i.e. for the training set). tau is normalized by the
    window length."""

    def __init__(self, windows, sensor_mean=None, sensor_std=None):
        self.windows = windows
        if sensor_mean is None or sensor_std is None:
            all_s = np.concatenate([w["x_s"] for w in windows], axis=0)
            sensor_mean = all_s.mean(axis=0, keepdims=True)
            sensor_std = all_s.std(axis=0, keepdims=True) + SENSOR_STD_EPS
        self.sensor_mean, self.sensor_std = sensor_mean, sensor_std

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        window_sec = float(w["window_sec"])
        return {
            "x_a": torch.tensor(w["x_a"], dtype=torch.float32),
            "x_s": torch.tensor((w["x_s"] - self.sensor_mean) / self.sensor_std, dtype=torch.float32),
            "tau_a": torch.tensor(w["tau_a"], dtype=torch.float32) / window_sec,
            "tau_s": torch.tensor(w["tau_s"], dtype=torch.float32) / window_sec,
            "label": torch.tensor(w["label"], dtype=torch.long),
        }


def split_fingerprint(windows):
    """(n windows, sum of raw sensor values, sum of sensor timestamps, class counts) — deterministic data check."""
    x_s = np.stack([w["x_s"] for w in windows]).astype(np.float64)
    tau_s = np.stack([w["tau_s"] for w in windows]).astype(np.float64)
    counts = tuple(np.bincount([w["label"] for w in windows], minlength=N_CLASSES).tolist())
    return (len(windows), round(float(x_s.sum()), 3), round(float(tau_s.sum()), 3), counts)


# --- 1. split + parse ---
mission_splits = split_missions(DATASET_ROOT)
split_dirs = {s: [m for label in sorted(mission_splits[s]) for m in mission_splits[s][label]] for s in mission_splits}
assert not set(split_dirs["train"]) & (set(split_dirs["val"]) | set(split_dirs["test"])), "LEAKAGE: mission in two splits"
parsed_missions = {m: parse_mission(m) for s in split_dirs for m in tqdm(split_dirs[s], desc=f"parse {s}", leave=False)}

# --- 2. strongest-64 bins from TRAINING audio only ---
bin_source_missions = [m for m in split_dirs["train"] if parsed_missions[m] is not None]
assert set(bin_source_missions) <= set(split_dirs["train"])
SELECTED_BINS = select_strongest_bins([parsed_missions[m] for m in bin_source_missions], N_AUDIO_BINS)
assert len(SELECTED_BINS) == N_AUDIO_BINS
print(f"[LEAKAGE CHECK] strongest-{N_AUDIO_BINS} bins computed from {len(bin_source_missions)} TRAINING missions only "
      f"(0 of {len(split_dirs['val']) + len(split_dirs['test'])} val/test missions) -> frozen for all splits: "
      f"{SELECTED_BINS.tolist()}")

# --- 3. windows (the same frozen bins for every split) ---
split_windows = {}
for split in ("train", "val", "test"):
    split_windows[split] = []
    for label in sorted(mission_splits[split]):
        for m in mission_splits[split][label]:
            if parsed_missions[m] is not None:
                split_windows[split].extend(slice_windows(parsed_missions[m], label, SELECTED_BINS))
    print(f"{split:<5}: {len(split_windows[split])} windows | audio {TA_AUDIO}x{N_AUDIO_BINS}, sensor {TS_SENSOR}x{N_SENSOR_CHANNELS}")
DATA_FINGERPRINT = {s: split_fingerprint(split_windows[s]) for s in split_windows}

# --- 4. datasets: sensor mean/std from the TRAINING split only, reused for val/test ---
train_ds = UAVWindowDataset(split_windows["train"])
val_ds = UAVWindowDataset(split_windows["val"], train_ds.sensor_mean, train_ds.sensor_std)
test_ds = UAVWindowDataset(split_windows["test"], train_ds.sensor_mean, train_ds.sensor_std)
_train_s = np.concatenate([w["x_s"] for w in split_windows["train"]], axis=0)
_val_s = np.concatenate([w["x_s"] for w in split_windows["val"]], axis=0)
assert val_ds.sensor_mean is train_ds.sensor_mean and test_ds.sensor_std is train_ds.sensor_std
assert np.array_equal(train_ds.sensor_mean, _train_s.mean(axis=0, keepdims=True))
print(f"[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | "
      f"train mean {np.round(train_ds.sensor_mean.ravel(), 3).tolist()} vs val's own mean "
      f"{np.round(_val_s.mean(axis=0), 3).tolist()} (not used)")
del _train_s, _val_s

# --- 5. real timestamps reach the model: tau must differ across windows ---
_tau_a = np.stack([train_ds[i]["tau_a"].numpy() for i in range(len(train_ds))])
_tau_s = np.stack([train_ds[i]["tau_s"].numpy() for i in range(len(train_ds))])
_n_unique_a, _n_unique_s = len(np.unique(_tau_a, axis=0)), len(np.unique(_tau_s, axis=0))
assert _n_unique_a > 1 and _n_unique_s > 1, "real timestamps are NOT reaching the model (tau identical for every window)"
assert not np.allclose(_tau_s, np.linspace(0.0, 1.0, TS_SENSOR)[None, :]), "tau_s equals a synthetic linspace grid"
print(f"[TIMESTAMP CHECK] {len(train_ds)} training windows: {_n_unique_a} distinct tau_a rows, {_n_unique_s} distinct "
      f"tau_s rows -> real per-window timestamps reach HTT/LACA ✅")
del _tau_a, _tau_s

Cloning https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git ...


Cloning into '/kaggle/working/tii_uav_dataset'...


Class 0 (19 flights) -> train 13, val 2, test 4
Class 1 (20 flights) -> train 14, val 3, test 3
Class 2 (20 flights) -> train 14, val 3, test 3
Class 3 (20 flights) -> train 14, val 3, test 3
Class 4 (20 flights) -> train 14, val 3, test 3


Updating files: 100% (217/217), done.


parse train:   0%|          | 0/69 [00:00<?, ?it/s]

parse val:   0%|          | 0/14 [00:00<?, ?it/s]

parse test:   0%|          | 0/16 [00:00<?, ?it/s]

[LEAKAGE CHECK] strongest-64 bins computed from 69 TRAINING missions only (0 of 30 val/test missions) -> frozen for all splits: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 501, 502, 503, 504, 505, 506, 507, 508, 509, 510, 511, 514, 515, 516, 517, 518, 519, 520, 521, 522, 523, 754, 757, 758, 759, 760, 761, 762, 763, 764, 765, 766]
train: 4934 windows | audio 64x64, sensor 128x6
val  : 954 windows | audio 64x64, sensor 128x6
test : 1130 windows | audio 64x64, sensor 128x6
[LEAKAGE CHECK] sensor standardization stats = TRAINING-split stats (same objects reused for val/test) | train mean [0.05299999937415123, -0.0430000014603138, -9.78600025177002, 0.004000000189989805, 0.003000000026077032, -0.0] vs val's own mean [0.029999999329447746, -0.061000000685453415, -9.748000144958496, 0.004999999888241291, 0.003000000026077032, -0.0] (not used)
[TIMESTAMP CHECK] 4934 training windows: 4934 distinct 

## 4. Model definition
Encoders → HTT → bi-directional LACA → 2 Mamba blocks → attention pooling → classifier. `AeroSyncMamba(beta=...)` passes β to both LACA directions. β is a plain float: it adds no parameters and draws no random numbers, so for a given seed every β value gets identical initial weights.

In [4]:
# ==============================================================================
# MODEL — encoders -> HTT (modality + continuous-time embedding of the REAL tau)
# -> bi-directional LACA (lag-aware cross-attention, mlp g(Δτ), β = per run (0.0 / 1.0), 4 heads)
# -> [H_a ; SEP ; H_s] -> 2 Mamba blocks -> attention pooling -> classifier.
# Auxiliary per-modality heads feed only the consistency loss.
# ==============================================================================
class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = TIME_EMBED_FREQUENCIES):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        return self.mlp(torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1))


class HeterogeneousTemporalTokenization(nn.Module):
    """token = LayerNorm(features + modality embedding + time embedding(tau)); modality 0 = audio, 1 = sensor."""

    def __init__(self, d_model: int):
        super().__init__()
        self.d_model = d_model
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        return self.norm(z + e_m + self.time_embedding(tau))


class PairwiseLACA(nn.Module):
    """Cross-attention from modality m to n with a learned lag bias g(Δτ) added to the scores (scaled by β)."""

    def __init__(self, d_model: int, num_heads: int = LACA_NUM_HEADS, beta: float = LACA_BETA):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, LAG_BIAS_HIDDEN), nn.Tanh(), nn.Linear(LAG_BIAS_HIDDEN, 1))

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)
        lag_bias = self.lag_bias(delta_tau).squeeze(-1).unsqueeze(1)

        attn = F.softmax(scores + self.beta * lag_bias, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)

        attn_mean = attn.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (attn_mean * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return self.out_proj(out), expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, beta: float = LACA_BETA):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, beta=beta)   # beta scales the lag bias in BOTH directions
        self.laca_sa = PairwiseLACA(d_model, beta=beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)
        return Ha + self.gamma_as * H_a_from_s, Hs + self.gamma_sa * H_s_from_a, {"lag_as": lag_as, "lag_sa": lag_sa}


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM."""

    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


class FallbackGatedConvBlock(nn.Module):
    """CPU-only stand-in when mamba_ssm is unavailable: depthwise causal conv + gated MLP. NOT a state-space
    model (no hidden state, no A/B/C, no discretization)."""

    def __init__(self, d_model: int, d_conv: int, expand: int):
        super().__init__()
        d_inner = expand * d_model
        self.in_proj = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(d_inner, d_inner, kernel_size=d_conv, padding=d_conv - 1, groups=d_inner, bias=True)
        self.mixer = nn.Sequential(nn.Linear(d_inner, d_inner), nn.GELU(), nn.Linear(d_inner, d_inner))
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        L = x.shape[1]
        x_branch, z_branch = self.in_proj(self.norm(x)).chunk(2, dim=-1)
        x_conv = F.silu(self.conv1d(x_branch.transpose(1, 2))[:, :, :L].transpose(1, 2))
        return x + self.out_proj(self.mixer(x_conv) * F.silu(z_branch))


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, expand=MAMBA_EXPAND):
    if MAMBA_CLASS is not None:
        return nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(num_blocks)])
    return nn.ModuleList([FallbackGatedConvBlock(d_model, d_conv, expand) for _ in range(num_blocks)])


# The reference results were produced by a model whose constructor drew, from the torch RNG, the initial weights
# of a 4-block (d_state 16, d_conv 4) Mamba stack before the final 2-block stack was built from the SAME starting
# RNG state. Every module built afterwards (pooling, classifier, auxiliary heads) and the whole training RNG stream
# (shuffle order, dropout masks) depend on the RNG state that 4-block draw leaves behind. To reproduce those
# results exactly, the constructor performs the same draw and discards it; the discarded blocks are never part of
# the model. This is a fixed step of weight initialization, not a configuration option.
RNG_REFERENCE_STACK = (4, 16, 4)   # (num_blocks, d_state, d_conv) whose initialization is drawn and discarded


class AeroSyncMamba(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_classes: int = N_CLASSES, beta: float = LACA_BETA):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.beta = float(beta)

        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.audio_encoder = nn.Sequential(
            nn.Linear(N_AUDIO_BINS, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
        self.htt = HeterogeneousTemporalTokenization(d_model)
        self.laca = BiModalLACAFusion(d_model, beta=self.beta)   # plain float: no params, no RNG draws
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        rng_before = torch.get_rng_state()
        build_fusion_blocks(d_model, *RNG_REFERENCE_STACK)
        rng_after_reference = torch.get_rng_state()
        torch.set_rng_state(rng_before)
        self.mamba_blocks = build_fusion_blocks(d_model, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        torch.set_rng_state(rng_after_reference)

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )
        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)
        H_tilde_a, H_tilde_s, lags = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        F_seq = torch.cat([H_tilde_a, self.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        for block in self.mamba_blocks:
            F_seq = block(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled = {"z_a": za_p, "z_s": zs_p}
        aux_logits = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled, aux_logits, lags


def verify_architecture(model, beta=LACA_BETA):
    """Finalized architecture, read from the constructed modules (beta = the value this model must use)."""
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        assert laca.num_heads == LACA_NUM_HEADS, f"{name}: {laca.num_heads} heads"
        assert laca.beta == beta, f"{name}: beta {laca.beta} != requested {beta}"
        assert sum(p.numel() for p in laca.lag_bias.parameters()) == LAG_BIAS_PARAMS
    assert len(model.mamba_blocks) == MAMBA_NUM_BLOCKS
    mamba_params = sum(p.numel() for p in model.mamba_blocks.parameters())
    if MAMBA_CLASS is not None:
        for block in model.mamba_blocks:
            m = block.mamba
            assert (m.d_state, m.d_conv, m.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
        assert mamba_params == EXPECTED_MAMBA_PARAMS, f"Mamba params {mamba_params:,} != {EXPECTED_MAMBA_PARAMS:,}"
    return mamba_params


_probe_counts = {}
for _b in BETA_VALUES:
    _probe = AeroSyncMamba(beta=_b)
    verify_architecture(_probe, beta=_b)
    _probe_counts[_b] = sum(p.numel() for p in _probe.parameters())
    del _probe
assert len(set(_probe_counts.values())) == 1, f"parameter count depends on beta: {_probe_counts}"
_probe = AeroSyncMamba()
print(f"Model: {_probe_counts[BETA_VALUES[0]]:,} params for every beta in {BETA_VALUES} | fusion blocks: "
      f"{type(_probe.mamba_blocks[0]).__name__} x {len(_probe.mamba_blocks)} ({verify_architecture(_probe):,} params) | "
      f"LACA {LACA_NUM_HEADS} heads, g(Δτ) {LAG_BIAS_PARAMS} params | backbone: {FUSION_BACKBONE}")
del _probe

Model: 1,688,788 params for every beta in [0.0, 1.0] | fusion blocks: MambaBlock x 2 (874,496 params) | LACA 4 heads, g(Δτ) 97 params | backbone: mamba_ssm.Mamba (real selective-scan SSM, CUDA)


## 5. Training function
`train_one_run(seed, beta)` resets every RNG with `seed` and builds a fresh model with `beta`. It checks that β reached both LACA modules and prints the β·g(Δτ) contribution before training and at the best checkpoint. At β = 0.0 that contribution must be exactly 0. It then trains, evaluates on the test set, and saves the test probabilities.

In [5]:
# ==============================================================================
# TRAINING — loss, metrics, the beta * lag_bias sanity check, and
# train_one_run(seed, beta): one full model end-to-end (fresh model / optimizer /
# scheduler, 20 epochs, best-validation checkpoint), evaluated on the test set;
# returns the trained model and its test-set softmax probabilities.
# ==============================================================================
class AeroSyncLoss(nn.Module):
    """L = CE_classweighted(logits, y) + LAMBDA_ALIGN * InfoNCE(z_a, z_s) + LAMBDA_CONS * KL-consistency."""

    def __init__(self, class_weights):
        super().__init__()
        self.lambda_align = LAMBDA_ALIGN
        self.lambda_cons = LAMBDA_CONS
        self.temperature = INFONCE_TEMPERATURE
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        sim = torch.matmul(F.normalize(z1, p=2, dim=-1), F.normalize(z2, p=2, dim=-1).t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_logits):
        loss_cls = self.ce(logits, targets)
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_logits["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_logits["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0
        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": loss_cls.item(), "loss_align": loss_align.item(), "loss_cons": loss_cons.item()}


def compute_metrics(y_true, y_prob):
    """accuracy, macro-F1, balanced accuracy, macro one-vs-rest AUROC from class probabilities (prediction = argmax)."""
    y_pred = y_prob.argmax(axis=1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_auroc": float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")),
    }


def train_class_weights(windows):
    """Inverse class frequency over the TRAINING windows."""
    counts = np.bincount([w["label"] for w in windows], minlength=N_CLASSES)
    return torch.tensor(counts.sum() / (len(counts) * np.maximum(counts, 1)), dtype=torch.float32)


def _batch_to_device(batch, non_blocking=False):
    return tuple(batch[k].to(device, non_blocking=non_blocking) for k in ("x_a", "x_s", "tau_a", "tau_s", "label"))


def predict_probs(model, loader):
    """Softmax probabilities and true labels over a loader (eval mode, no grad), in loader order."""
    model.eval()
    probs, labels = [], []
    with torch.no_grad():
        for batch in loader:
            xa, xs, ta, ts, y = _batch_to_device(batch)
            logits, _, _, _ = model(xa, xs, ta, ts)
            probs.append(F.softmax(logits, dim=-1).cpu().numpy())
            labels.append(y.cpu().numpy())
    return np.concatenate(probs), np.concatenate(labels)


train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=NUM_WORKERS)
CLASS_WEIGHTS = train_class_weights(split_windows["train"]).to(device)
TEST_LABELS = np.array([w["label"] for w in split_windows["test"]], dtype=np.int64)   # test-window order, fixed


# Fixed probe batch for the lag-bias check: first BATCH_SIZE training windows, read by index (no DataLoader,
# no RNG consumed).
_probe_items = [train_ds[i] for i in range(min(BATCH_SIZE, len(train_ds)))]
LAG_PROBE = tuple(torch.stack([p[k] for p in _probe_items]) for k in ("x_a", "x_s", "tau_a", "tau_s"))
del _probe_items


def lag_bias_contribution(model):
    """Recompute, with the model's OWN modules, exactly the term PairwiseLACA adds to the attention scores,
    beta * lag_bias(Δτ), for both LACA directions (same check as the v9 ablation). Runs in eval mode under
    no_grad inside fork_rng, so it cannot change the RNG streams, BatchNorm statistics or weights."""
    xa, xs, ta, ts = (t.to(device) for t in LAG_PROBE)
    was_training = model.training
    out = {}
    cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
    with torch.random.fork_rng(devices=cuda_devices), torch.no_grad():
        model.eval()
        za = model.audio_encoder(xa)
        zs = model.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)
        Ha = model.htt(za, ta, modality_idx=0)
        Hs = model.htt(zs, ts, modality_idx=1)
        for name, laca, tau_m, tau_n in (("laca_as", model.laca.laca_as, ta, ts),
                                         ("laca_sa", model.laca.laca_sa, ts, ta)):
            delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)
            B_mn = laca.lag_bias(delta_tau).squeeze(-1).unsqueeze(1)
            contrib = laca.beta * B_mn
            out[name] = {"beta": laca.beta, "lag_bias_absmax": float(B_mn.abs().max()),
                         "contrib_absmax": float(contrib.abs().max()),
                         "contrib_finite": bool(torch.isfinite(contrib).all())}
    model.train(was_training)
    return out


def report_lag_bias_contribution(model, beta, tag, when):
    """Print beta * lag_bias(...).abs().max() for both directions; assert exactly 0 when beta == 0.0.
    Returns the larger of the two contributions."""
    res = lag_bias_contribution(model)
    for name, r in res.items():
        print(f"{tag} [LAG-BIAS {when}] {name}: beta={r['beta']} | max|lag_bias(Δτ)| = {r['lag_bias_absmax']:.6f} | "
              f"max|beta * lag_bias(Δτ)| = {r['contrib_absmax']!r}")
    if beta == 0.0:
        for name, r in res.items():
            assert r["contrib_finite"] and r["contrib_absmax"] == 0.0, (
                f"ABLATION FAILED: {name} lag-bias contribution is {r['contrib_absmax']!r}, not exactly 0")
        print(f"{tag} [LAG-BIAS {when}] ✅ beta=0.0 ABLATION CONFIRMED: beta * lag_bias(Δτ) is EXACTLY 0.0 in both "
              f"LACA directions -> content-only cross-attention (timestamp embeddings still in every token)")
    else:
        for name, r in res.items():
            assert r["contrib_absmax"] > 0.0, f"{name}: beta={beta} but the lag-bias contribution is 0"
    return max(r["contrib_absmax"] for r in res.values())


def train_one_run(seed: int, beta: float):
    """Train one model with `seed` applied to every RNG and LACA scale `beta`, evaluate it on the test set.
    Returns (trained model on CPU, result dict incl. test probabilities)."""
    t0 = time.time()
    tag = f"[seed={seed} beta={beta}]"
    set_all_seeds(seed)
    model = AeroSyncMamba(beta=beta).to(device)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()
    rng_sha1 = hashlib.sha1(torch.get_rng_state().numpy().tobytes()).hexdigest()
    mamba_params = verify_architecture(model, beta=beta)
    applied = (model.laca.laca_as.beta, model.laca.laca_sa.beta)
    assert applied == (beta, beta), f"beta NOT applied to the model: {applied} != {beta}"
    print(f"{tag} beta in effect, read back from the constructed model: laca_as.beta = {applied[0]} | "
          f"laca_sa.beta = {applied[1]} ✅")
    print(f"{tag} seeds applied | initial-weights fingerprint {init_sha1[:12]} | torch RNG after "
          f"construction {rng_sha1[:12]} | fusion {type(model.mamba_blocks[0]).__name__} ({mamba_params:,} params)")
    report_lag_bias_contribution(model, beta, tag, "before training")

    criterion = AeroSyncLoss(CLASS_WEIGHTS)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)

    best_val_f1, best_state, best_epoch = 0.0, None, None
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = defaultdict(float)
        for batch in tqdm(train_loader, desc=f"seed {seed} beta {beta} epoch {epoch:02d}/{EPOCHS}", leave=False):
            xa, xs, ta, ts, y = _batch_to_device(batch, non_blocking=True)
            optimizer.zero_grad()
            logits, pooled, aux_logits, _ = model(xa, xs, ta, ts)
            loss, parts = criterion(logits, y, pooled, aux_logits)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            running["loss"] += loss.item()
            for k, v in parts.items():
                running[k] += v
        scheduler.step()

        val_probs, val_labels = predict_probs(model, val_loader)
        val_f1 = compute_metrics(val_labels, val_probs)["macro_f1"]
        improved = val_f1 > best_val_f1
        if improved:
            best_val_f1, best_state, best_epoch = val_f1, copy.deepcopy(model.state_dict()), epoch
        n = len(train_loader)
        print(f"{tag} epoch {epoch:02d}/{EPOCHS} | train loss {running['loss'] / n:.4f} (cls "
              f"{running['loss_cls'] / n:.4f}, align {running['loss_align'] / n:.4f}, cons {running['loss_cons'] / n:.4f})"
              f" | val macro-F1 {val_f1:.4f}{'  * best' if improved else ''}")

    model.load_state_dict(best_state)
    lag_contrib_best = report_lag_bias_contribution(model, beta, tag, "best checkpoint")
    test_probs, test_labels = predict_probs(model, test_loader)
    metrics = compute_metrics(test_labels, test_probs)
    run_name = f"beta{beta}_seed{seed}"
    torch.save(best_state, CHECKPOINT_DIR / f"{run_name}_best.pt")
    np.save(PROBS_DIR / f"{run_name}_test_probs.npy", test_probs)
    np.save(PROBS_DIR / f"{run_name}_test_labels.npy", test_labels)

    result = {
        "beta": beta,
        "seed": seed,
        **metrics,
        "best_val_macro_f1": round(best_val_f1, 6),
        "best_epoch": best_epoch,
        "n_test_windows": len(test_labels),
        "init_sha1": init_sha1[:12],
        "rng_sha1": rng_sha1[:12],
        "lag_contrib_absmax_best": lag_contrib_best,
        "train_seconds": round(time.time() - t0, 1),
        "test_probs": test_probs,
        "test_labels": test_labels,
    }
    return model.cpu(), result

## 6. Paired training driver: 10 runs
For each seed in 42, 123, 2024, 7, 99, the driver runs β = 0.0 and then β = 1.0 back to back. After every run it appends a row to `/kaggle/working/beta_paired_replication_results.csv` and prints a progress line. The seed-42 / β = 1.0 reference check runs immediately after run 2, before the remaining 8 runs. When a pair finishes, the driver checks that its two runs shared identical initial weights and RNG streams.

In [6]:
# ==============================================================================
# PAIRED DRIVER — for seed in SEEDS: beta=0.0 then beta=1.0. After every run:
# print metrics, append a CSV row, keep test probabilities, free GPU, print
# progress. Seed 42 / beta=1.0 is checked against the v16 baseline immediately
# after run 2 (it gates the remaining 8 runs). Each finished pair is checked for
# identical initial weights + RNG stream (only beta differs).
# ==============================================================================
RESULTS_CSV_FIELDS = ["run", "beta", "seed", "accuracy", "macro_f1", "balanced_accuracy", "macro_auroc",
                      "best_val_macro_f1", "best_epoch", "n_test_windows", "init_sha1", "rng_sha1",
                      "lag_contrib_absmax_best", "train_seconds", "finished_at"]
METRICS = ["accuracy", "macro_f1", "balanced_accuracy", "macro_auroc"]

if RESULTS_CSV.exists():
    RESULTS_CSV.rename(RESULTS_CSV.with_suffix(f".{datetime.now():%Y%m%d_%H%M%S}.bak.csv"))


def check_seed42_baseline(r):
    """seed 42 / beta=1.0 vs the v16 baseline, every metric within ±SAME_SEED_TOLERANCE. Returns (pass, max|delta|)."""
    print("\n" + "#" * 100 + "\n#  REFERENCE CHECK — seed 42 / beta=1.0 vs the v16 baseline (±"
          f"{SAME_SEED_TOLERANCE} per metric)\n" + "#" * 100)
    worst = 0.0
    for k, ref in REFERENCE_SEED42.items():
        d = r[k] - ref
        worst = max(worst, abs(d))
        print(f"#  {k:<18} reference {ref:.4f} | this run {r[k]:.4f} | delta {d:+.4f}"
              + ("" if abs(d) <= SAME_SEED_TOLERANCE else "   <-- OUTSIDE TOLERANCE"))
    rng_ok = r["rng_sha1"].startswith(REFERENCE_SEED42_RNG_PREFIX)
    print(f"#  torch RNG after construction {r['rng_sha1']} (reference prefix {REFERENCE_SEED42_RNG_PREFIX}): "
          f"{'match' if rng_ok else 'MISMATCH'}")
    ok = worst <= SAME_SEED_TOLERANCE
    if ok:
        print(f"#  ✅ PASS — seed 42 / beta=1.0 reproduces the v16 baseline (max |delta| {worst:.4f}). "
              f"Proceeding with the remaining {len(RUN_ORDER) - 2} runs.")
    else:
        print("🚨" * 45)
        print(f"🚨  seed 42 / beta=1.0 DOES NOT REPRODUCE the v16 baseline (max |delta| {worst:.4f} > {SAME_SEED_TOLERANCE}).")
        print("🚨  The code path, data or environment differs from v16 — see the diagnostics in Section 8.")
        print("🚨" * 45)
    return ok, worst


def check_seed42_beta0_vs_v9(r):
    ref = V9_SEED42_MACRO_F1[0.0]
    d = r["macro_f1"] - ref
    print(f"\n[v9 COMPARISON] seed 42 / beta=0.0 macro-F1 {r['macro_f1']:.4f} vs v9 {ref:.4f} (delta {d:+.4f})")
    if abs(d) <= SAME_SEED_TOLERANCE:
        print("[v9 COMPARISON] ✅ matches v9 within ±0.0005")
    else:
        print("⚠️ " * 30)
        print(f"⚠️  seed 42 / beta=0.0 does NOT match v9's 0.8909 (delta {d:+.4f}). Some difference is EXPECTED: the"
              f" configuration changed after v9 (v9's own beta=1.0 baseline was {V9_SEED42_MACRO_F1[1.0]:.4f}; v16's is"
              f" {REFERENCE_SEED42['macro_f1']:.4f}, mainly from the v12 Mamba configuration). A shift about as large as"
              f" the beta=1.0 shift ({REFERENCE_SEED42['macro_f1'] - V9_SEED42_MACRO_F1[1.0]:+.4f}) is unsurprising;"
              f" a much larger one is worth investigating.")
        print("⚠️ " * 30)


run_results = {}
BASELINE_PASS, BASELINE_MAX_DELTA = None, None
t_driver = time.time()
for i, (seed, beta) in enumerate(RUN_ORDER, start=1):
    print("\n" + "=" * 100 + f"\nRUN {i}/{len(RUN_ORDER)} — seed={seed}, beta={beta}"
          + ("  (lag-bias ABLATION)" if beta == 0.0 else "  (finalized lag bias)") + "\n" + "=" * 100)
    model, result = train_one_run(seed, beta)
    run_results[(seed, beta)] = result
    print(f"[seed={seed} beta={beta}] TEST: accuracy {result['accuracy']:.4f} | macro-F1 {result['macro_f1']:.4f} | "
          f"balanced accuracy {result['balanced_accuracy']:.4f} | macro-AUROC {result['macro_auroc']:.4f} | best epoch "
          f"{result['best_epoch']} | {result['train_seconds']:.0f}s")

    write_header = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_CSV_FIELDS)
        if write_header:
            writer.writeheader()
        writer.writerow({"run": i, **{k: result[k] for k in RESULTS_CSV_FIELDS if k in result},
                         "finished_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")})

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    if seed == 42 and beta == 0.0:
        check_seed42_beta0_vs_v9(result)
    if seed == 42 and beta == 1.0:
        BASELINE_PASS, BASELINE_MAX_DELTA = check_seed42_baseline(result)
        if not BASELINE_PASS and HALT_ON_BASELINE_MISMATCH:
            raise RuntimeError("seed 42 / beta=1.0 did not reproduce the v16 baseline — stopping before the remaining "
                               "runs (set HALT_ON_BASELINE_MISMATCH = False in Section 2 to continue anyway).")

    # --- pair complete? only beta may differ between its two runs ---
    if all((seed, b) in run_results for b in BETA_VALUES):
        r0, r1 = run_results[(seed, 0.0)], run_results[(seed, 1.0)]
        assert r0["init_sha1"] == r1["init_sha1"], f"seed {seed}: beta=0/1 initial weights differ — not a paired design"
        assert r0["rng_sha1"] == r1["rng_sha1"], f"seed {seed}: beta=0/1 RNG streams differ — not a paired design"
        assert np.array_equal(r0["test_labels"], r1["test_labels"]), f"seed {seed}: test windows differ"
        print(f"[PAIR seed={seed}] ✅ identical initial weights ({r0['init_sha1']}) and RNG stream ({r0['rng_sha1']}) "
              f"for beta=0.0 and beta=1.0 -> only beta differs | macro-F1 beta=0 {r0['macro_f1']:.4f}, beta=1 "
              f"{r1['macro_f1']:.4f}, paired diff (1 - 0) {r1['macro_f1'] - r0['macro_f1']:+.4f}")

    n_pairs = sum(all((s, b) in run_results for b in BETA_VALUES) for s in SEEDS)
    elapsed = time.time() - t_driver
    print(f"\n>>> {i}/{len(RUN_ORDER)} complete: seed={seed}, beta={beta} finished | complete pairs {n_pairs}/{len(SEEDS)} | "
          f"elapsed {elapsed / 60:.1f} min | est. remaining {elapsed / i * (len(RUN_ORDER) - i) / 60:.1f} min")

# --- different seeds genuinely differ (distinct initial weights and RNG streams) ---
init_prints = {s: run_results[(s, 1.0)]["init_sha1"] for s in SEEDS}
rng_prints = {s: run_results[(s, 1.0)]["rng_sha1"] for s in SEEDS}
assert len(set(init_prints.values())) == len(SEEDS), f"two seeds produced identical initial weights: {init_prints}"
assert len(set(rng_prints.values())) == len(SEEDS), f"two seeds produced identical RNG streams: {rng_prints}"
print(f"\n[SEED CHECK] {len(SEEDS)} seeds -> distinct initial weights / RNG streams; each shared by its beta=0/1 pair: "
      + ", ".join(f"{s}: {init_prints[s]}/{rng_prints[s]}" for s in SEEDS))
print(f"All {len(RUN_ORDER)} runs finished in {(time.time() - t_driver) / 60:.1f} min. Results: {RESULTS_CSV}")


RUN 1/10 — seed=42, beta=0.0  (lag-bias ABLATION)
[seed=42 beta=0.0] beta in effect, read back from the constructed model: laca_as.beta = 0.0 | laca_sa.beta = 0.0 ✅
[seed=42 beta=0.0] seeds applied | initial-weights fingerprint a92fd4248184 | torch RNG after construction 5641b401720d | fusion MambaBlock (874,496 params)
[seed=42 beta=0.0] [LAG-BIAS before training] laca_as: beta=0.0 | max|lag_bias(Δτ)| = 0.042988 | max|beta * lag_bias(Δτ)| = 0.0
[seed=42 beta=0.0] [LAG-BIAS before training] laca_sa: beta=0.0 | max|lag_bias(Δτ)| = 0.465048 | max|beta * lag_bias(Δτ)| = 0.0
[seed=42 beta=0.0] [LAG-BIAS before training] ✅ beta=0.0 ABLATION CONFIRMED: beta * lag_bias(Δτ) is EXACTLY 0.0 in both LACA directions -> content-only cross-attention (timestamp embeddings still in every token)


seed 42 beta 0.0 epoch 01/20:   0%|          | 0/155 [00:00<?, ?it/s]

[seed=42 beta=0.0] epoch 01/20 | train loss 1.4174 (cls 1.0707, align 3.1025, cons 0.3650) | val macro-F1 0.8261  * best


seed 42 beta 0.0 epoch 02/20:   0%|          | 0/155 [00:00<?, ?it/s]

[seed=42 beta=0.0] epoch 02/20 | train loss 0.9259 (cls 0.5977, align 2.6650, cons 0.6171) | val macro-F1 0.6814


...

seed 99 beta 1.0 epoch 19/20:   0%|          | 0/155 [00:00<?, ?it/s]

[seed=99 beta=1.0] epoch 19/20 | train loss 0.2892 (cls 0.1179, align 1.2753, cons 0.4375) | val macro-F1 0.8822


seed 99 beta 1.0 epoch 20/20:   0%|          | 0/155 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d99881f80e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d99881f80e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

[seed=99 beta=1.0] epoch 20/20 | train loss 0.2911 (cls 0.1209, align 1.2657, cons 0.4364) | val macro-F1 0.8884  * best
[seed=99 beta=1.0] [LAG-BIAS best checkpoint] laca_as: beta=1.0 | max|lag_bias(Δτ)| = 0.178239 | max|beta * lag_bias(Δτ)| = 0.17823916673660278
[seed=99 beta=1.0] [LAG-BIAS best checkpoint] laca_sa: beta=1.0 | max|lag_bias(Δτ)| = 0.363377 | max|beta * lag_bias(Δτ)| = 0.3633767366409302
[seed=99 beta=1.0] TEST: accuracy 0.8726 | macro-F1 0.8675 | balanced accuracy 0.8646 | macro-AUROC 0.9840 | best epoch 20 | 202s
[PAIR seed=99] ✅ identical initial weights (7e3cd71f848e) and RNG stream (81dcd4638a88) for beta=0.0 and beta=1.0 -> only beta differs | macro-F1 beta=0 0.8638, beta=1 0.8675, paired diff (1 - 0) +0.0037

>>> 10/10 complete: seed=99, beta=1.0 finished | complete pairs 5/5 | elapsed 43.5 min | est. remaining 0.0 min

[SEED CHECK] 5 seeds -> distinct initial weights / RNG streams; each shared by its beta=0/1 pair: 42: a92fd4248184/5641b401720d, 123: 79ccb8c584

## 7. Paired comparison & statistical verdict
This section reads `beta_paired_replication_results.csv` and uses only seeds with both β values, so it also works on a partial run. It prints one row per seed for each of the four metrics. For the primary metric, macro-F1, it also computes:
- the mean paired difference;
- the std of the paired differences;
- the win count;
- a **paired t-test (primary)** and a **Wilcoxon signed-rank test (secondary)**.

**Why the t-test is primary.** With 5 pairs, the exact two-sided Wilcoxon p-value can never go below 2/2⁵ = 0.0625, even when all 5 seeds agree. So it can never reach 0.05. The sign test (the win count) has the same floor. The paired t-test uses the size of each difference, not just its sign or rank, so it can reach a low p-value with 5 pairs. Its assumption is that the differences are roughly normal. That cannot be checked with n = 5, so the Wilcoxon test is reported alongside it as a robustness check.

**Verdict rule.** β = 1.0's advantage counts as **WELL-SUPPORTED** only if all three hold:
1. β = 1 wins on at least n − 1 seeds (4 of 5);
2. the mean paired difference is positive;
3. the mean paired difference exceeds the std of the paired differences (equivalently t > √n ≈ 2.24), **or** the paired t-test gives p < 0.05.

Otherwise the verdict is **NOT WELL-SUPPORTED**. If β = 0.0 meets the mirror-image criteria, the verdict is **REVERSED**.

In [7]:
# ==============================================================================
# PAIRED ANALYSIS — per-seed paired table (4 metrics), paired statistics on the
# macro-F1 differences, and the verdict. Reads the CSV, so it covers exactly the
# complete (beta=0, beta=1) pairs that finished.
# ==============================================================================
import pandas as pd
from scipy import stats

df = pd.read_csv(RESULTS_CSV)
complete_seeds = [s for s in SEEDS if set(df.loc[df["seed"] == s, "beta"]) >= set(BETA_VALUES)]
n = len(complete_seeds)
print(f"Complete (beta=0, beta=1) pairs: {n}/{len(SEEDS)} -> seeds {complete_seeds}")
if n < len(SEEDS):
    print(f"⚠️  PARTIAL RESULT: only {n} of {len(SEEDS)} pairs finished — the statistics below use those pairs only.")
assert n >= 2, "need at least 2 complete pairs for any paired statistic"

pivot = {m: df.pivot_table(index="seed", columns="beta", values=m).loc[complete_seeds] for m in METRICS}

# --- 1. per-seed paired tables, all 4 metrics ---
for m in METRICS:
    p = pivot[m]
    d = p[1.0] - p[0.0]
    print(f"\n{'PRIMARY — ' if m == 'macro_f1' else ''}{m}")
    print(f"  {'seed':>6} {'beta=0.0':>10} {'beta=1.0':>10} {'diff (1-0)':>11}  favors")
    for s in complete_seeds:
        fav = "beta=1" if d[s] > 0 else ("beta=0" if d[s] < 0 else "tie")
        print(f"  {s:>6} {p.loc[s, 0.0]:>10.4f} {p.loc[s, 1.0]:>10.4f} {d[s]:>+11.4f}  {fav}")
    print(f"  {'mean':>6} {p[0.0].mean():>10.4f} {p[1.0].mean():>10.4f} {d.mean():>+11.4f}  "
          f"(std of paired diffs {d.std(ddof=1):.4f}; beta=1 wins {int((d > 0).sum())}/{n})")

# --- 2. paired statistics on the macro-F1 differences ---
f1_b0 = pivot["macro_f1"][0.0].to_numpy()
f1_b1 = pivot["macro_f1"][1.0].to_numpy()
d = f1_b1 - f1_b0
mean_d = float(d.mean())
sd_d = float(d.std(ddof=1))
se_d = sd_d / math.sqrt(n)
wins, losses, ties = int((d > 0).sum()), int((d < 0).sum()), int((d == 0).sum())
t_res = stats.ttest_rel(f1_b1, f1_b0)
t_stat, p_t = float(t_res.statistic), float(t_res.pvalue)
t_crit = float(stats.t.ppf(0.975, df=n - 1))
ci = (mean_d - t_crit * se_d, mean_d + t_crit * se_d)
try:
    w_res = stats.wilcoxon(d, alternative="two-sided", method="exact")
    p_w, w_note = float(w_res.pvalue), ""
except Exception as exc:   # e.g. every difference exactly zero
    p_w, w_note = float("nan"), f" (not computable: {exc})"
p_sign = float(stats.binomtest(wins, wins + losses, 0.5).pvalue) if wins + losses > 0 else float("nan")
min_wilcoxon_p = 2.0 / 2 ** n

print("\n" + "=" * 100)
print(f"PAIRED STATISTICS — test macro-F1, beta=1.0 minus beta=0.0, {n} matched seeds")
print("=" * 100)
print(f"  paired differences          : " + ", ".join(f"{s}: {x:+.4f}" for s, x in zip(complete_seeds, d)))
print(f"  mean paired difference      : {mean_d:+.4f}")
print(f"  std of paired differences   : {sd_d:.4f}   (sample std, ddof=1; the relevant noise for a paired design)")
print(f"  mean / std                  : {mean_d / sd_d if sd_d > 0 else float('inf'):+.2f}   (> 1 required)")
print(f"  95% CI of mean difference   : [{ci[0]:+.4f}, {ci[1]:+.4f}]  (t, df={n - 1})")
print(f"  win count                   : beta=1 better on {wins}/{n} seeds (beta=0 better on {losses}, ties {ties})")
print(f"  paired t-test (PRIMARY)     : t = {t_stat:+.3f}, df = {n - 1}, two-sided p = {p_t:.4f}")
print(f"  Wilcoxon signed-rank (exact): two-sided p = {p_w:.4f}{w_note}   (smallest achievable with n={n}: "
      f"{min_wilcoxon_p:.4f})")
print(f"  sign test (win count)       : two-sided p = {p_sign:.4f}")
print(f"\n  context (unpaired view): beta=0 mean {f1_b0.mean():.4f} ± {f1_b0.std(ddof=1):.4f} | beta=1 mean "
      f"{f1_b1.mean():.4f} ± {f1_b1.std(ddof=1):.4f} | measured seed-to-seed std (v15/v16) {SEED_TO_SEED_STD_F1:.4f}")
if 42 in complete_seeds:
    d42 = d[complete_seeds.index(42)]
    print(f"  seed 42 alone: paired diff {d42:+.4f} here vs v9's single-seed {V9_SINGLE_SEED_BETA_EFFECT:+.4f}")
for s in complete_seeds:
    v16 = V16_BETA1_MACRO_F1.get(s)
    if v16 is not None:
        x = float(pivot["macro_f1"].loc[s, 1.0])
        print(f"  beta=1 seed {s:<5} {x:.4f} vs v16 {v16:.4f} (delta {x - v16:+.4f})"
              + ("" if abs(x - v16) <= SAME_SEED_TOLERANCE else "   <-- differs from v16"))

# --- 3. verdict ---
exceeds_noise = abs(mean_d) > sd_d
significant = p_t < PAIRED_ALPHA
beta1_consistent = mean_d > 0 and wins >= n - 1
beta0_consistent = mean_d < 0 and losses >= n - 1
if beta1_consistent and (exceeds_noise or significant):
    VERDICT = "WELL-SUPPORTED"
elif beta0_consistent and (exceeds_noise or significant):
    VERDICT = "REVERSED"
else:
    VERDICT = "NOT WELL-SUPPORTED"

print("\n" + "#" * 100)
print(f"#  VERDICT: beta=1.0's advantage over beta=0.0 is {VERDICT}")
print("#" * 100)
print(f"#  criteria: direction consistent (>= {n - 1}/{n} seeds favor beta=1): "
      f"{'YES' if beta1_consistent else 'NO'} ({wins}/{n}) | |mean diff| > std of paired diffs: "
      f"{'YES' if exceeds_noise else 'NO'} ({abs(mean_d):.4f} vs {sd_d:.4f}) | paired t-test p < {PAIRED_ALPHA}: "
      f"{'YES' if significant else 'NO'} (p = {p_t:.4f})")
if VERDICT == "WELL-SUPPORTED":
    print(f"#  The original single-seed finding HOLDS UP. beta=1.0 beats beta=0.0 on {wins} of {n} matched seeds, by "
          f"{mean_d:+.4f} macro-F1 on average, and that mean exceeds the paired noise ({sd_d:.4f}). The lag bias "
          f"is doing measurable work; beta=1.0 moves from PROVISIONAL to REAL.")
elif VERDICT == "REVERSED":
    print(f"#  The original single-seed finding DOES NOT HOLD UP; it is REVERSED. beta=0.0 (no lag bias) beats "
          f"beta=1.0 on {losses} of {n} matched seeds, by {-mean_d:+.4f} macro-F1 on average, exceeding the paired "
          f"noise ({sd_d:.4f}). The lag bias is hurting in this configuration.")
else:
    reasons = []
    if not (beta1_consistent or beta0_consistent):
        reasons.append(f"the direction is inconsistent across seeds (beta=1 wins {wins}, beta=0 wins {losses})")
    if not (exceeds_noise or significant):
        reasons.append(f"the mean paired difference ({mean_d:+.4f}) does not exceed the paired noise ({sd_d:.4f}) "
                       f"and p = {p_t:.4f} >= {PAIRED_ALPHA}")
    print(f"#  The original single-seed finding (+{V9_SINGLE_SEED_BETA_EFFECT:.4f}) DOES NOT HOLD UP: "
          + "; ".join(reasons) + ". This data does not show that the lag bias improves macro-F1. beta=1.0 may be "
          "kept as a default, but it is not an established finding.")
if n < len(SEEDS):
    print(f"#  ⚠️  based on {n}/{len(SEEDS)} pairs only.")
print("#" * 100)

Complete (beta=0, beta=1) pairs: 5/5 -> seeds [42, 123, 2024, 7, 99]

accuracy
    seed   beta=0.0   beta=1.0  diff (1-0)  favors
      42     0.9088     0.9080     -0.0009  beta=0
     123     0.8876     0.8876     +0.0000  tie
    2024     0.8805     0.8823     +0.0018  beta=1
       7     0.9062     0.9062     +0.0000  tie
      99     0.8690     0.8726     +0.0035  beta=1
    mean     0.8904     0.8913     +0.0009  (std of paired diffs 0.0018; beta=1 wins 2/5)

PRIMARY — macro_f1
    seed   beta=0.0   beta=1.0  diff (1-0)  favors
      42     0.9057     0.9048     -0.0010  beta=0
     123     0.8827     0.8819     -0.0008  beta=0
    2024     0.8764     0.8783     +0.0020  beta=1
       7     0.9019     0.9021     +0.0001  beta=1
      99     0.8638     0.8675     +0.0037  beta=1
    mean     0.8861     0.8869     +0.0008  (std of paired diffs 0.0020; beta=1 wins 3/5)

balanced_accuracy
    seed   beta=0.0   beta=1.0  diff (1-0)  favors
      42     0.9030     0.9022     -0.0008  b

## 8. Final validation
A recap of the reference checks: seed 42 / β = 1.0 against the v16 baseline (with data and RNG diagnostics), seed 42 / β = 0.0 against v9, pair integrity, and the verdict.

In [8]:
# ==============================================================================
# FINAL VALIDATION — reference reproduction + diagnostics, pair integrity, verdict
# ==============================================================================
print("=" * 100)
print("FINAL VALIDATION")
print("=" * 100)

r42 = run_results[(42, 1.0)]
print("\n(1) seed 42 / beta=1.0 vs the v16 baseline (tolerance ±0.0005 per metric)")
for k, ref in REFERENCE_SEED42.items():
    print(f"    {k:<18} reference {ref:.4f} | this run {r42[k]:.4f} | delta {r42[k] - ref:+.4f}")
print("    diagnostics:")
data_checks = {
    "strongest-64 bin indices": SELECTED_BINS.tolist() == REFERENCE_DATA["bin_indices"],
    **{f"{s} windows / sensor values / timestamps / class counts": DATA_FINGERPRINT[s] ==
       REFERENCE_DATA["sensor_fingerprint"][s] for s in ("train", "val", "test")},
    "seed-42 torch RNG after model construction": r42["rng_sha1"].startswith(REFERENCE_SEED42_RNG_PREFIX),
    "fusion backbone is real mamba_ssm": MAMBA_CLASS is not None,
}
for name, ok in data_checks.items():
    print(f"      {name:<58}: {'match' if ok else 'MISMATCH'}")

r42_b0 = run_results[(42, 0.0)]
print(f"\n(2) seed 42 / beta=0.0 vs v9 (informational): {r42_b0['macro_f1']:.4f} vs {V9_SEED42_MACRO_F1[0.0]:.4f} "
      f"(delta {r42_b0['macro_f1'] - V9_SEED42_MACRO_F1[0.0]:+.4f})")

pairs_ok = all(run_results[(s, 0.0)]["init_sha1"] == run_results[(s, 1.0)]["init_sha1"] and
               run_results[(s, 0.0)]["rng_sha1"] == run_results[(s, 1.0)]["rng_sha1"]
               for s in SEEDS if (s, 0.0) in run_results and (s, 1.0) in run_results)
ablation_ok = all(r["lag_contrib_absmax_best"] == 0.0 for (s, b), r in run_results.items() if b == 0.0)

print("\n" + "=" * 100)
print(f"  seed 42 / beta=1.0 reproduction : {'✅ PASS' if BASELINE_PASS else '❌ FAIL'} (max |delta| "
      f"{BASELINE_MAX_DELTA:.4f}, tolerance {SAME_SEED_TOLERANCE})")
print(f"  seed 42 / beta=0.0 vs v9 0.8909 : {r42_b0['macro_f1']:.4f} "
      f"({'match' if abs(r42_b0['macro_f1'] - V9_SEED42_MACRO_F1[0.0]) <= SAME_SEED_TOLERANCE else 'differs — see Section 6 note'})")
print(f"  pairs share weights + RNG       : {'✅' if pairs_ok else '❌'} ({n} complete pairs)")
print(f"  beta=0 lag-bias term exactly 0  : {'✅' if ablation_ok else '❌'} (all beta=0.0 runs, before training and at "
      f"the best checkpoint)")
print(f"  VERDICT (macro-F1, paired)      : beta=1.0 advantage {VERDICT} — mean diff {mean_d:+.4f}, std of diffs "
      f"{sd_d:.4f}, wins {wins}/{n}, paired t-test p = {p_t:.4f}")
print("=" * 100)
if not BASELINE_PASS:
    print("🚨" * 45)
    print("🚨  seed 42 / beta=1.0 did NOT reproduce the v16 baseline. If a data diagnostic above is a MISMATCH the")
    print("🚨  preprocessing differs; if only the RNG line differs, model construction changed; if all diagnostics")
    print("🚨  match, the difference arose in training (GPU / library version). The paired comparison is still")
    print("🚨  internally matched, but it is not tied to the v16 reference numbers.")
    print("🚨" * 45)

FINAL VALIDATION

(1) seed 42 / beta=1.0 vs the v16 baseline (tolerance ±0.0005 per metric)
    accuracy           reference 0.9080 | this run 0.9080 | delta -0.0000
    macro_f1           reference 0.9048 | this run 0.9048 | delta -0.0000
    balanced_accuracy  reference 0.9022 | this run 0.9022 | delta -0.0000
    macro_auroc        reference 0.9894 | this run 0.9894 | delta -0.0000
    diagnostics:
      strongest-64 bin indices                                  : match
      train windows / sensor values / timestamps / class counts : match
      val windows / sensor values / timestamps / class counts   : match
      test windows / sensor values / timestamps / class counts  : match
      seed-42 torch RNG after model construction                : match
      fusion backbone is real mamba_ssm                         : match

(2) seed 42 / beta=0.0 vs v9 (informational): 0.9057 vs 0.8909 (delta +0.0148)

  seed 42 / beta=1.0 reproduction : ✅ PASS (max |delta| 0.0000, tolerance 0.0005)
